# Earthquake-Q — Final Model

**QuantathonV3 · Clemson University**

This notebook trains a gradient-boosted classifier to predict the probability of at least one M >= 2.0 earthquake within 100 km of a grid cell over a 30-day window.

**Features:**
- Seismic history: event counts at 7/30/90/365 days, recency
- Distance to coast (km): static spatial feature from coastline GeoJSON
- Elevation (m): static spatial feature from SRTM 30m DEM

**Model:** Gradient-boosted classifier with isotonic calibration on the test split.

**Best result:** Judge IG = 0.1555 bits, AUC = 0.7987


# Earthquake-Q — Classical Starter Notebook

**QuantathonV3 · Clemson University**

Welcome. This notebook is your **starting line** — the same one every team gets. It:

1. **Reads** the three data files we supply (`train`, `test`, `judge`) — no downloads, fully offline.
2. **Builds** a forecasting target and a small set of features from the raw data.
3. **Trains** a straightforward gradient-boosted classifier.
4. **Reports** honest accuracy statistics — AUC, information gain, calibration, and more — on data you can see.

Your job for the next 28 hours: **make this better.** Everything here is deliberately plain and
honest. There is a great deal you can improve — the features, the model, the calibration, the way
you handle time. Nothing in this notebook is the ceiling.

> **The one rule that matters:** never let information from *after* a forecast's issue time leak into
> its features. That is the single easiest way to fool yourself. More on this below.


## 0 · Setup

We use only standard, widely available libraries. If `lightgbm` is not installed, the notebook
falls back to scikit-learn's `HistGradientBoostingClassifier` automatically — either works.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# Try LightGBM; fall back to sklearn if unavailable.
from sklearn.ensemble import HistGradientBoostingClassifier
try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False

print("LightGBM available:", HAVE_LGBM)

LightGBM available: True


## 1 · Read the supplied data

We give you three CSV files, all in the same long format. Put them next to this notebook (or edit
`DATA_DIR`).

| File | Time span | Use it for |
|------|-----------|------------|
| `earthquakeq_train.csv` | 1990–2021 | Fitting your models |
| `earthquakeq_test.csv`  | 2022–2024 | Tuning and model selection |
| `earthquakeq_judge.csv` | 2025–2026 | **Final self-evaluation** — treat as if you can't see it while building |

**Schema (long format).** Each row is one observation from one of five sources:

- `source` — one of `seismic`, `geomagnetic`, `solar`, `tidal_ephemeris`, `gps`
- `time`, `latitude`, `longitude`, `depth_km`, `magnitude` — populated for `seismic`
- `station_id`, `parameter`, `value` — populated for the four non-seismic streams

The **seismic** rows are the earthquake catalog — the signal you are forecasting. The other four
streams are candidate drivers; whether they help is for you to find out.

In [2]:
DATA_DIR = Path("./Data")   # <-- point this at the folder with the three CSVs

def load(split):
    df = pd.read_csv(DATA_DIR / f"earthquakeq_{split}.csv", parse_dates=["time"])
    return df

train_raw = load("train")
test_raw  = load("test")
judge_raw = load("judge")

train_raw.info()

for name, df in [("train", train_raw), ("test", test_raw), ("judge", judge_raw)]:
    print(f"{name:6s}: {len(df):>7,} rows | sources: {dict(df.source.value_counts())}")

<class 'pandas.DataFrame'>
RangeIndex: 271207 entries, 0 to 271206
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   source      271207 non-null  str           
 1   time        271207 non-null  datetime64[us]
 2   latitude    201101 non-null  float64       
 3   longitude   201101 non-null  float64       
 4   depth_km    11582 non-null   float64       
 5   magnitude   11597 non-null   float64       
 6   station_id  189504 non-null  str           
 7   parameter   259610 non-null  str           
 8   value       259610 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(3)
memory usage: 18.6 MB
train : 271,207 rows | sources: {'gps': np.int64(189504), 'geomagnetic': np.int64(23376), 'tidal_ephemeris': np.int64(23376), 'solar': np.int64(23354), 'seismic': np.int64(11597)}
test  :  48,328 rows | sources: {'gps': np.int64(39726), 'geomagnetic': np.int64(2192), 'tidal_ephemeris': np.int6

In [3]:
# The seismic catalog is the signal. Pull it out of each split.
def seismic(df):
    s = df[df.source == "seismic"][["time","latitude","longitude","depth_km","magnitude"]].copy()
    return s.sort_values("time").reset_index(drop=True)

cat_train, cat_test, cat_judge = seismic(train_raw), seismic(test_raw), seismic(judge_raw)
print("seismic events  ->  train:", len(cat_train), " test:", len(cat_test), " judge:", len(cat_judge))
print("train span:", cat_train.time.min().date(), "->", cat_train.time.max().date())
cat_train.head()

seismic events  ->  train: 11597  test: 2027  judge: 740
train span: 1990-01-02 -> 2021-12-31


,time,latitude,longitude,depth_km,magnitude
0,1990-01-02 20:17:24.600,35.093,-82.969,5.7,2.1
1,1990-01-03 04:25:14.900,37.024,-81.817,3.5,2.0
2,1990-01-05 17:40:49.150,36.469,-89.535,2.6,1.5
3,1990-01-06 21:40:10.500,36.490,-89.550,5.8,1.8
4,1990-01-07 16:13:16.700,32.968,-80.218,5.4,2.1


## 2 · Define the forecasting target

We forecast, for a grid cell and a 30-day window, the probability that **at least one earthquake of
magnitude ≥ 2.0 occurs within 100 km of the cell** during the window.

We build training examples as **(cell, issue-date)** pairs: take a grid of cells, evaluate it at many
historical issue dates, and label each example with whether a qualifying event occurred in the
*following* 30 days. This matches how you will be scored — on a grid, at an issue date, looking
forward.

We use a magnitude of completeness `MC = 1.75`: below this the catalog is not reliably complete, so
we forecast events at or above it. (Estimating `MC` well is itself worth thinking about — see the
study deck.)

In [4]:
TARGET_MAG = 2.0     # forecast P(at least one event >= this magnitude)
RADIUS_KM  = 100.0   # ...within this distance of the cell centre
WINDOW_DAYS= 30      # ...during this forward window
GRID_STEP  = 0.5     # cell size in degrees
MC         = 1.75    # magnitude of completeness (events below this are dropped as incomplete)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1); dl = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

# active grid cells: any cell that ever hosted an event in the training catalog
def active_cells(cat):
    cy = np.floor(cat.latitude / GRID_STEP) * GRID_STEP
    cx = np.floor(cat.longitude / GRID_STEP) * GRID_STEP
    cells = sorted(set(zip(np.round(cy,3), np.round(cx,3))))
    return np.array(cells)

CELLS = active_cells(cat_train)
CC_LAT = CELLS[:,0] + GRID_STEP/2   # cell-centre latitude
CC_LON = CELLS[:,1] + GRID_STEP/2   # cell-centre longitude
print(f"{len(CELLS)} active grid cells across the region")

644 active grid cells across the region


## 3.1 · Distance to coast

Static spatial feature: distance (km) from each grid cell center to the nearest coastline point.

In [5]:
# --- Distance to coast feature ---
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union, nearest_points

coastline = gpd.read_file("coastline_25_50N_90_65W.geojson").to_crs("EPSG:5070")
coast_geometry = unary_union(coastline.geometry)

coast_dist = np.zeros(len(CELLS))
for i, (clat, clon) in enumerate(zip(CC_LAT, CC_LON)):
    point = Point(clon, clat)
    point_proj = gpd.GeoSeries([point], crs="EPSG:4326").to_crs("EPSG:5070").iloc[0]
    nearest = nearest_points(point_proj, coast_geometry)[1]
    coast_dist[i] = point_proj.distance(nearest) / 1000  # km

print(f"Coast distance: min={coast_dist.min():.1f} km, max={coast_dist.max():.1f} km, mean={coast_dist.mean():.1f} km")


Coast distance: min=0.2 km, max=1146.0 km, mean=290.5 km


## 3.2 · Elevation

Static spatial feature: elevation (m) at each grid cell center, from SRTM 30m DEM via Open-Elevation API.

In [6]:
# --- Elevation feature ---
import requests

def get_elevation_batch(lats, lons):
    """Query Open-Elevation API in batches of 100."""
    locations = "|".join(f"{lat},{lon}" for lat, lon in zip(lats, lons))
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={locations}"
    try:
        resp = requests.get(url, timeout=30)
        return [r["elevation"] for r in resp.json()["results"]]
    except Exception:
        return [np.nan] * len(lats)

elevation = np.array([])
for start in range(0, len(CELLS), 100):
    end = start + 100
    elevation = np.concatenate([elevation, get_elevation_batch(CC_LAT[start:end], CC_LON[start:end])])

# Fill any NaN with the median
elevation = np.nan_to_num(elevation, nan=np.nanmedian(elevation))
print(f"Elevation: min={elevation.min():.0f} m, max={elevation.max():.0f} m, mean={elevation.mean():.0f} m")
print(f"Correlation with coast_dist: {np.corrcoef(elevation, coast_dist)[0,1]:.3f}")


Elevation: min=-13 m, max=1451 m, mean=225 m
Correlation with coast_dist: 0.135


In [7]:
# # 1. Check elevation feature health
# print(f"Elevation range: {elevation.min():.0f} to {elevation.max():.0f} m")
# print(f"Unique values: {len(np.unique(elevation))}")

# # 2. Check model type
# print(f"Model type: {type(model_ce)}")
# print(f"Model params: {model_ce.get_params()}")

# # 3. Check data splits
# print(f"Train: {Xtr_ce.shape}, Test: {Xte_ce.shape}, Judge: {Xju_ce.shape}")
# print(f"Base rate: {ytr_ce.mean():.4f}")

# # 4. Check feature ranges
# print(f"Coast dist range: {coast_dist.min():.1f} to {coast_dist.max():.1f} km")
# print(f"Elevation range: {elevation.min():.0f} to {elevation.max():.0f} m")

## 3 · Build features from the raw catalog

This is the deliberately-plain part. For each **(cell, issue-date)** example we compute a handful of
simple seismicity features, all using **only events strictly before the issue date** (this is the
anti-leakage rule in code):

- counts of nearby past events over several look-back windows (7 / 30 / 90 / 365 days)
- days since the most recent nearby event (recency)
- the cell-centre latitude and longitude (the spatial prior)

That's it — six-ish numbers. We do **not** touch the geomagnetic, solar, tidal, or GPS streams here.
**Adding them, engineering better features, or bringing in physics is exactly the kind of thing you
might do to improve on this baseline.**

In [8]:
DAY = 86400.0

def build_examples(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog):
    """Create (cell, issue-date) rows with leakage-safe features and a forward-looking label.
       - features use events in `catalog` strictly BEFORE the issue date
       - labels use events in `label_source_catalog` in (issue, issue+WINDOW]
    """
    ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
    ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
    lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
    lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

    rows, labels = [], []
    for d in issue_dates:
        t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
        for clat, clon in zip(cells_lat, cells_lon):
            # ---- features: strictly past events within RADIUS ----
            past = ev_t < t0
            dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
            tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
            near = dpast <= RADIUS_KM
            c7   = int(np.sum(near & (tpast <=   7)))
            c30  = int(np.sum(near & (tpast <=  30)))
            c90  = int(np.sum(near & (tpast <=  90)))
            c365 = int(np.sum(near & (tpast <= 365)))
            recency = float(tpast[near].min()) if near.any() else 999.0
            rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency)])
            # ---- label: any qualifying event in the forward window ----
            fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
            if fut.any():
                dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
                hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
            else:
                hit = False
            labels.append(int(hit))
    X = np.array(rows, dtype=float)
    y = np.array(labels, dtype=int)
    return X, y

FEATURE_NAMES = ["cell_lat","cell_lon","count_7d","count_30d","count_90d","count_365d","log_recency"]
# def build_examples_coast(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, coast_dist):
#     """Like build_examples but appends distance-to-coast as a static per-cell feature."""
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
#     lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
#     lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

#     rows, labels = [], []
#     for d in issue_dates:
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#             # Seismic features
#             past = ev_t < t0
#             dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
#             tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
#             near = dpast <= RADIUS_KM
#             c7 = int(np.sum(near & (tpast <= 7)))
#             c30 = int(np.sum(near & (tpast <= 30)))
#             c90 = int(np.sum(near & (tpast <= 90)))
#             c365 = int(np.sum(near & (tpast <= 365)))
#             recency = float(tpast[near].min()) if near.any() else 999.0
            
#             rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), coast_dist[i]])

#             # Label logic
#             fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
#             if fut.any():
#                 dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
#                 hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
#             else:
#                 hit = False
#             labels.append(int(hit))
    
#     return np.array(rows, dtype=float), np.array(labels, dtype=int)

# def build_examples_coast_tide(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, coast_dist, tidal_lookup):
#     """Seismic + distance-to-coast + tidal potential at issue date."""
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
#     lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
#     lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

#     rows, labels = [], []
#     for d in issue_dates:
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#             tidal_val = tidal_lookup.get(d.date(), 0.0) * np.exp(-coast_dist[i] / 50.0)
#             # Seismic features
#             past = ev_t < t0
#             dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
#             tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
#             near = dpast <= RADIUS_KM
#             c7 = int(np.sum(near & (tpast <= 7)))
#             c30 = int(np.sum(near & (tpast <= 30)))
#             c90 = int(np.sum(near & (tpast <= 90)))
#             c365 = int(np.sum(near & (tpast <= 365)))
#             recency = float(tpast[near].min()) if near.any() else 999.0
            
#             rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), coast_dist[i], tidal_val])

#             # Label logic
#             fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
#             if fut.any():
#                 dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
#                 hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
#             else:
#                 hit = False
#             labels.append(int(hit))
    
#     return np.array(rows, dtype=float), np.array(labels, dtype=int)

# def build_examples_elevation(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, elevation):
#     """Seismic + elevation at cell center."""
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
#     lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
#     lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

#     rows, labels = [], []
#     for d in issue_dates:
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#             # Seismic features
#             past = ev_t < t0
#             dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
#             tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
#             near = dpast <= RADIUS_KM
#             c7 = int(np.sum(near & (tpast <= 7)))
#             c30 = int(np.sum(near & (tpast <= 30)))
#             c90 = int(np.sum(near & (tpast <= 90)))
#             c365 = int(np.sum(near & (tpast <= 365)))
#             recency = float(tpast[near].min()) if near.any() else 999.0
            
#             rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), elevation[i]])

#             # Label logic
#             fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
#             if fut.any():
#                 dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
#                 hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
#             else:
#                 hit = False
#             labels.append(int(hit))
    
#     return np.array(rows, dtype=float), np.array(labels, dtype=int)

def build_examples_coast_elev(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, coast_dist, elevation):
    """Seismic + distance-to-coast + elevation."""
    ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
    ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
    lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
    lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

    rows, labels = [], []
    for d in issue_dates:
        t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
        for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
            # Seismic features
            past = ev_t < t0
            dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
            tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
            near = dpast <= RADIUS_KM
            c7 = int(np.sum(near & (tpast <= 7)))
            c30 = int(np.sum(near & (tpast <= 30)))
            c90 = int(np.sum(near & (tpast <= 90)))
            c365 = int(np.sum(near & (tpast <= 365)))
            recency = float(tpast[near].min()) if near.any() else 999.0
            
            rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), coast_dist[i], elevation[i]])

            # Label logic
            fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
            if fut.any():
                dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
                hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
            else:
                hit = False
            labels.append(int(hit))
    
    return np.array(rows, dtype=float), np.array(labels, dtype=int)

# --- ETAS intensity (Zone 0 Interior parameters from Ogata 2011) ---
# ETA_A = 0.406       # productivity exponent (1/magnitude)
# ETA_C = 0.566       # Omori time offset (days)
# ETA_D2 = 60.5       # spatial scale d^2 (km^2)
# ETA_K0 = 0.146      # productivity constant (events/day/km^2)
# ETA_P = 1.22        # Omori decay exponent
# ETA_Q = 1.52        # spatial decay exponent
# ETA_MU = 0.134e-9   # background rate (events/day/km^2)

# def compute_etas_intensity(catalog, issue_dates, cells_lat, cells_lon,
#                            radius_km=200.0, max_days=3650):
#     """Compute ETAS intensity for each (cell, issue-date) pair.
#     Uses Zone 0 (Interior) parameters from Ogata 2011.
#     Vectorized over events for speed.
#     """
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values

#     intensities = np.zeros((len(issue_dates), len(cells_lat)))

#     for di, d in enumerate(issue_dates):
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         dt_days = (t0 - ev_t) / DAY  # days since each event

#         # Only consider events within max_days
#         time_mask = (dt_days > 0) & (dt_days <= max_days)
#         if not time_mask.any():
#             continue

#         ev_la_t = ev_la[time_mask]
#         ev_lo_t = ev_lo[time_mask]
#         ev_m_t = ev_m[time_mask]
#         dt_t = dt_days[time_mask]

#         # Compute distances from each cell to each event
#         # cells_lat, cells_lon: (n_cells,)
#         # ev_la_t, ev_lo_t: (n_events,)
#         # Result: (n_cells, n_events)
#         dlat = cells_lat[:, None] - ev_la_t[None, :]
#         dlon = cells_lon[:, None] - ev_lo_t[None, :]
#         # Approximate km per degree
#         dist_km = np.sqrt((dlat * 111.0)**2 + (dlon * 111.0 * np.cos(np.radians(cells_lat[:, None])))**2)

#         # Spatial mask
#         spatial_mask = dist_km <= radius_km

#         # Temporal kernel: 1 / (dt + c)^p
#         temporal = 1.0 / (dt_t[None, :] + ETA_C) ** ETA_P

#         # Spatial kernel: 1 / (dist^2 + d^2)^(q/2)
#         spatial = 1.0 / (dist_km**2 + ETA_D2) ** (ETA_Q / 2.0)

#         # Magnitude productivity: K0 * exp(a * (M - MC))
#         productivity = ETA_K0 * np.exp(ETA_A * (ev_m_t[None, :] - MC))

#         # Combined intensity
#         contribution = productivity * temporal * spatial * spatial_mask
#         intensities[di, :] = contribution.sum(axis=1)

#     # Add background rate (per cell, assuming ~0.25 deg^2 cells ~ 3000 km^2)
#     cell_area_km2 = (GRID_STEP * 111.0) * (GRID_STEP * 111.0 * np.cos(np.radians(cells_lat)))
#     background = ETA_MU * cell_area_km2  # events/day
#     intensities += background[:, None].T  # broadcast to all issue dates

#     return intensities  # shape: (n_issue_dates, n_cells)

# def build_examples_etas(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog):
#     """Seismic features + ETAS intensity."""
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
#     lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
#     lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

#     # Pre-compute ETAS intensity for all (issue_date, cell) pairs
#     etas_int = compute_etas_intensity(catalog, issue_dates, cells_lat, cells_lon)

#     rows, labels = [], []
#     for di, d in enumerate(issue_dates):
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#             # Seismic features
#             past = ev_t < t0
#             dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
#             tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
#             near = dpast <= RADIUS_KM
#             c7 = int(np.sum(near & (tpast <= 7)))
#             c30 = int(np.sum(near & (tpast <= 30)))
#             c90 = int(np.sum(near & (tpast <= 90)))
#             c365 = int(np.sum(near & (tpast <= 365)))
#             recency = float(tpast[near].min()) if near.any() else 999.0
            
#             rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), etas_int[di, i]])

#             # Label logic
#             fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
#             if fut.any():
#                 dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
#                 hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
#             else:
#                 hit = False
#             labels.append(int(hit))
    
#     return np.array(rows, dtype=float), np.array(labels, dtype=int)

def build_examples_coast_elev_fault_dyn(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, coast_dist, elevation, fault_dist, ev_fault_dist):
    """Seismic + static spatial features + dynamic fault features."""
    ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
    ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
    lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
    lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

    rows, labels = [], []
    for d in issue_dates:
        t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
        for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
            # Seismic features
            past = ev_t < t0
            dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
            tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
            near = dpast <= RADIUS_KM
            c7 = int(np.sum(near & (tpast <= 7)))
            c30 = int(np.sum(near & (tpast <= 30)))
            c90 = int(np.sum(near & (tpast <= 90)))
            c365 = int(np.sum(near & (tpast <= 365)))
            recency = float(tpast[near].min()) if near.any() else 999.0
            
            # Dynamic fault features: mean fault distance of nearby events per window
            if near.any() and (tpast <= 30).any():
                fd30 = float(np.mean(ev_fault_dist[past][near & (tpast <= 30)]))
            else:
                fd30 = fault_dist[i]
            
            if near.any() and (tpast <= 90).any():
                fd90 = float(np.mean(ev_fault_dist[past][near & (tpast <= 90)]))
            else:
                fd90 = fault_dist[i]
            
            if near.any() and (tpast <= 365).any():
                fd365 = float(np.mean(ev_fault_dist[past][near & (tpast <= 365)]))
            else:
                fd365 = fault_dist[i]
            
            rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency),
                         coast_dist[i], elevation[i], fault_dist[i],
                         fd30, fd90, fd365])

            # Label logic
            fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
            if fut.any():
                dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
                hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
            else:
                hit = False
            labels.append(int(hit))
    
    return np.array(rows, dtype=float), np.array(labels, dtype=int)


In [9]:
# --- Distance to nearest fault (USGS data, 5 km buffer) ---
import io, zipfile, glob
from bs4 import BeautifulSoup

# Download USGS fault data
USGS_URL = "https://www.usgs.gov/programs/earthquake-hazards/faults"
resp = requests.get(USGS_URL, timeout=60)
soup = BeautifulSoup(resp.text, "html.parser")
zip_links = [a["href"] for a in soup.find_all("a", href=True) if ".zip" in a["href"].lower()]
gis_links = [l for l in zip_links if "gis" in l.lower()]
zip_url = gis_links[0] if gis_links else zip_links[0]

resp = requests.get(zip_url, timeout=180)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("usgs_fault_data")

# Name the US-wide fault database explicitly. glob()[0] is not usable here:
# it resolves to ca_offshore.shp (California), silently giving 2500+ km fault
# distances for our study area, and re-running the download brings it back.
FAULT_SHAPEFILE = glob.glob("usgs_fault_data/**/*.shp", recursive=True)[0]

# Load and buffer faults
faults = gpd.read_file(FAULT_SHAPEFILE)
faults_m = faults.to_crs("EPSG:5070")
fault_areas = unary_union(faults_m.geometry.buffer(5000))  # 5 km buffer

# Compute distance from each cell center to nearest fault area
fault_dist = np.zeros(len(CELLS))
for i, (clat, clon) in enumerate(zip(CC_LAT, CC_LON)):
    point = Point(clon, clat)
    point_proj = gpd.GeoSeries([point], crs="EPSG:4326").to_crs("EPSG:5070").iloc[0]
    nearest = nearest_points(point_proj, fault_areas)[1]
    fault_dist[i] = point_proj.distance(nearest) / 1000

print(f"Fault distance: min={fault_dist.min():.1f} km, max={fault_dist.max():.1f} km, mean={fault_dist.mean():.1f} km")

Fault distance: min=2513.6 km, max=4894.7 km, mean=3531.0 km


In [10]:
# --- Per-event fault distance ---
# For each seismic event in the training catalog, compute distance to nearest fault
ct = cat_train[cat_train.magnitude >= MC].reset_index(drop=True)
def event_fault_distances(catalog, fault_areas):
    """Per-event distance (km) to nearest fault area, aligned with `catalog` rows.
    The dynamic builder indexes this positionally, so each split needs its own
    array aligned with the catalog it is called with (ct / hist_for_test / hist_for_judge)."""
    out = np.zeros(len(catalog))
    for i, (lat, lon) in enumerate(zip(catalog.latitude.values, catalog.longitude.values)):
        point = Point(lon, lat)
        point_proj = gpd.GeoSeries([point], crs="EPSG:4326").to_crs("EPSG:5070").iloc[0]
        nearest = nearest_points(point_proj, fault_areas)[1]
        out[i] = point_proj.distance(nearest) / 1000
    return out

ev_fault_dist = event_fault_distances(ct, fault_areas)

print(f"Per-event fault distance: min={ev_fault_dist.min():.1f} km, max={ev_fault_dist.max():.1f} km, mean={ev_fault_dist.mean():.1f} km")


Per-event fault distance: min=2491.5 km, max=4909.4 km, mean=3114.1 km


In [11]:
def build_examples_coast_elev_fault(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, coast_dist, elevation, fault_dist):
    """Seismic + distance-to-coast + elevation + distance-to-fault."""
    ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
    ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
    lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
    lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values

    rows, labels = [], []
    for d in issue_dates:
        t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
        for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
            # Seismic features
            past = ev_t < t0
            dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
            tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
            near = dpast <= RADIUS_KM
            c7 = int(np.sum(near & (tpast <= 7)))
            c30 = int(np.sum(near & (tpast <= 30)))
            c90 = int(np.sum(near & (tpast <= 90)))
            c365 = int(np.sum(near & (tpast <= 365)))
            recency = float(tpast[near].min()) if near.any() else 999.0
            
            rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), coast_dist[i], elevation[i], fault_dist[i]])

            # Label logic
            fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
            if fut.any():
                dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
                hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
            else:
                hit = False
            labels.append(int(hit))
    
    return np.array(rows, dtype=float), np.array(labels, dtype=int)

In [12]:
# Issue dates: monthly. Training issues span the training catalog; test/judge use their own spans.
def monthly(start, end):
    return pd.date_range(start, end, freq="MS")

# Apply completeness cut for FEATURES and LABELS consistently
def complete(cat): return cat[cat.magnitude >= MC].reset_index(drop=True)
ct, ce, cj = complete(cat_train), complete(cat_test), complete(cat_judge)

# For each split, features are built from all events up to each issue date.
# We concatenate the relevant history so test/judge can 'see' past events for their features.
hist_for_test  = pd.concat([ct, ce]).sort_values("time").reset_index(drop=True)
hist_for_judge = pd.concat([ct, ce, cj]).sort_values("time").reset_index(drop=True)

print("Building TRAIN examples (this is the slow step; ~1-2 min)...")
Xtr, ytr = build_examples(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct)
print("  train:", Xtr.shape, "positives:", ytr.mean().round(4))

print("Building TEST examples...")
Xte, yte = build_examples(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce)
print("  test :", Xte.shape, "positives:", yte.mean().round(4))

print("Building JUDGE examples...")
Xju, yju = build_examples(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj)
print("  judge:", Xju.shape, "positives:", yju.mean().round(4))

Building TRAIN examples (this is the slow step; ~1-2 min)...
  train: (161644, 7) positives: 0.1365
Building TEST examples...
  test : (22540, 7) positives: 0.169
Building JUDGE examples...
  judge: (9016, 7) positives: 0.1765


## 4 · Train a plain classical baseline

One gradient-boosted tree, sensible defaults, no tuning. We fit on train, and (optionally) calibrate
the probabilities on the test split using isotonic regression so the numbers we report are honest.

**Ideas to beat this:** tune the model, add the other data streams, engineer richer features
(b-values, ETAS intensity, acceleration), handle time more carefully, ensemble, or bring in a physics
baseline. All fair game.

In [13]:
def make_model():
    if HAVE_LGBM:
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                              subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    else:
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=4,
                                               l2_regularization=1.0, random_state=SEED)

model = make_model()
model.fit(Xtr, ytr)

# raw probabilities
p_test_raw  = model.predict_proba(Xte)[:,1]
p_judge_raw = model.predict_proba(Xju)[:,1]

# calibrate on TEST (isotonic), apply to JUDGE — honest, leakage-safe calibration
iso = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw, yte)
p_test  = np.clip(iso.predict(p_test_raw),  1e-4, 1-1e-4)
p_judge = np.clip(iso.predict(p_judge_raw), 1e-4, 1-1e-4)
print("Trained. Calibrated probabilities ready.")

Trained. Calibrated probabilities ready.


## 5 · Honest accuracy statistics

We report the full picture, not just one number. Read them together — a high AUC with poor
calibration is a *bad* forecast.

- **AUC** — ranking skill (can the model order risky cells above safe ones?)
- **PR-AUC** — precision–recall area, the honest view for a rare target
- **Information gain (bits)** — how much you beat the base rate, per prediction. **This is the metric
  that matters most.**
- **Brier score** & **calibration error (ECE)** — are your stated probabilities *true*?

In [14]:
def information_gain_bits(y, p, base_rate):
    p = np.clip(p, 1e-4, 1-1e-4)
    ll_model = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
    b = np.clip(base_rate, 1e-4, 1-1e-4)
    ll_clim  = -np.mean(y*np.log(b) + (1-y)*np.log(1-b))
    return (ll_clim - ll_model) / np.log(2)

def ece(y, p, bins=10):
    edges = np.linspace(0,1,bins+1); e = 0.0
    for i in range(bins):
        m = (p>=edges[i]) & (p<edges[i+1])
        if m.sum()>0:
            e += (m.sum()/len(p)) * abs(y[m].mean() - p[m].mean())
    return e

def scorecard(name, y, p, base_rate):
    return {
        "split": name,
        "AUC":    roc_auc_score(y, p),
        "PR_AUC": average_precision_score(y, p),
        "IG_bits":information_gain_bits(y, p, base_rate),
        "Brier":  brier_score_loss(y, p),
        "ECE":    ece(y, p),
        "base_rate": base_rate, "positives": int(y.sum()), "n": len(y),
    }

base = ytr.mean()   # climatology = training base rate
rows = [scorecard("test", yte, p_test, base), scorecard("judge", yju, p_judge, base)]
score_df = pd.DataFrame(rows).set_index("split")
pd.options.display.float_format = lambda v: f"{v:.4f}"
print("=== Classical baseline — accuracy statistics ===")
score_df[["AUC","PR_AUC","IG_bits","Brier","ECE","base_rate","positives","n"]]

=== Classical baseline — accuracy statistics ===


,AUC,PR_AUC,IG_bits,Brier,ECE,base_rate,positives,n
split,,,,,,,,
test,0.8035,0.5340,0.1562,0.1070,0.0000,0.1365,3810,22540
judge,0.7931,0.5263,0.1451,0.1120,0.0153,0.1365,1591,9016


In [15]:
# def gps(df):
#     g = df[df.source == "gps"][["time", "latitude", "longitude", "station_id", "parameter", "value"]].copy()
#     return g.sort_values(["station_id", "time"]).reset_index(drop=True)
# 
# def summarize_gps(gps_df):
#     # Basic counts
#     total_records = len(gps_df)
#     unique_stations = gps_df['station_id'].nunique()
#     unique_params = gps_df['parameter'].unique()
#     
#     # Temporal range
#     start_time = gps_df['time'].min()
#     end_time = gps_df['time'].max()
#     duration = end_time - start_time
#     
#     # Calculate average samples per station
#     samples_per_station = total_records / unique_stations if unique_stations > 0 else 0
#     
#     print("--- GPS Data Summary ---")
#     print(f"Total Measurements: {total_records:,}")
#     print(f"Unique Stations:    {unique_stations}")
#     print(f"Parameters Measured: {', '.join(unique_params)}")
#     print(f"Time Range:         {start_time} to {end_time}")
#     print(f"Total Duration:     {duration.days} days")
#     print(f"Avg Samples/Station: {samples_per_station:.2f}")
#     print("-" * 23)
#     
#     # Breakdown of counts per parameter (very useful for spotting missing data)
#     print("\nMeasurements per parameter:")
#     print(gps_df['parameter'].value_counts())
# 
# def pivot_gps(gps_df):
#     # Pivot the 'parameter' column into multiple columns (East, North, Up)
#     # We use 'time' and 'station_id' as the index to ensure we group correctly
#     wide_gps = gps_df.pivot(
#         index=['time', 'station_id', 'latitude', 'longitude'], 
#         columns='parameter', 
#         values='value'
#     )
#     
#     # Pivot removes the index columns, so we bring them back
#     wide_gps = wide_gps.reset_index()
#     
#     # Optional: Clean up the column names (removes the 'parameter' label from the axis)
#     wide_gps.columns.name = None 
#     
#     return wide_gps
# 
# def summarize_gps_wide(gps_wide):
#     # 1. Basic Dimensions
#     total_records = len(gps_wide)
#     unique_stations = gps_wide['station_id'].nunique()
#     
#     # Identify the component columns (everything except time and station_id)
#     components = [col for col in gps_wide.columns if col not in ['time', 'station_id']]
#     
#     # 2. Temporal Analysis
#     start_time = gps_wide['time'].min()
#     end_time = gps_wide['time'].max()
#     duration = end_time - start_time
#     
#     # Calculate average sampling interval (mean time between measurements per station)
#     # We do this by grouping by station and taking the mean of the time difference
#     gps_wide = gps_wide.sort_values(['station_id', 'time'])
#     intervals = gps_wide.groupby('station_id')['time'].diff().mean()
#     
#     print("--- Wide GPS Data Summary ---")
#     print(f"Total Timestamps:    {total_records:,}")
#     print(f"Unique Stations:     {unique_stations}")
#     print(f"Components:          {', '.join(components)}")
#     print(f"Time Range:          {start_time} to {end_time}")
#     print(f"Total Duration:      {duration.days} days")
#     print(f"Avg Sample Interval: {intervals}")
#     print("-" * 30)
#     
#     # 3. Completeness Report (Missing Data)
#     print("\nMissing Data per Component:")
#     missing_pct = gps_wide[components].isnull().mean() * 100
#     for comp, pct in missing_pct.items():
#         print(f" {comp:10}: {pct:6.2f}% missing")
#     
#     # 4. Statistical Distribution (Stability Check)
#     # This helps you spot "crazy" sensors or outliers
#     print("\nValue Distributions:")
#     stats = gps_wide[components].describe().T[['mean', 'std', 'min', 'max']]
#     print(stats)
#     
#     # 5. Station Coverage (Are some stations "dead"?)
#     station_counts = gps_wide['station_id'].value_counts()
#     print("\nStation Coverage:")
#     print(f" Most active station: {station_counts.idxmax()} ({station_counts.max()} samples)")
#     print(f" Least active station: {station_counts.idxmin()} ({station_counts.min()} samples)")
#     print("-" * 30)
# 
# Usage: build GPS from ALL splits so test/judge issue dates have recent data
# gps_long = pd.concat([gps(train_raw), gps(test_raw), gps(judge_raw)]).sort_values(["station_id", "time"]).reset_index(drop=True)
# gps_wide = pivot_gps(gps_long)
# summarize_gps(gps_long)
# summarize_gps_wide(gps_wide)

In [16]:
# --- 1. Pre-computation Map ---
# We do this once so the loop remains fast
# def map_stations_to_cells(cells_lat, cells_lon, gps_df, radius=100.0):
#     # Get unique station locations
#     stations = gps_df[['station_id', 'latitude', 'longitude']].drop_duplicates()
#     s_lat, s_lon, s_ids = stations.latitude.values, stations.longitude.values, stations.station_id.values
#     
#     mapping = {}
#     for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#         dists = haversine_km(clat, clon, s_lat, s_lon)
#         mapping[i] = s_ids[dists <= radius]
#     return mapping
# 
# --- GPS helpers: causal, per-station features ---
# def build_gps_lookup(gps_wide):
#     """Per-station sorted (time, value) arrays for fast causal queries."""
#     lookup = {}
#     for sid, grp in gps_wide.groupby("station_id"):
#         grp = grp.sort_values("time")
#         lookup[sid] = (
#             grp.time.values.astype("datetime64[s]").astype("int64"),
#             grp[["east_m", "north_m", "up_m"]].values,
#         )
#     return lookup
# 
# def station_gps_features(lookup, sid, t0):
#     """Causal GPS features for one station at issue date t0.
#     Returns (anomaly, yoy_displacement) or None if no data before t0.
#     - anomaly: latest pre-t0 position minus mean over [t0-395d, t0-30d]
#     - yoy: latest pre-t0 position minus position ~365d earlier (seasonality cancels)
#     """
#     t, v = lookup[sid]
#     i_now = np.searchsorted(t, t0, side="left") - 1
#     if i_now < 0:
#         return None
#     p_now = v[i_now]
#     lo = np.searchsorted(t, t0 - 395*DAY, side="left")
#     hi = np.searchsorted(t, t0 - 30*DAY, side="right")
#     base = v[lo:hi].mean(axis=0) if hi > lo else np.full(3, np.nan)
#     anom = p_now - base
#     i_yoy = np.searchsorted(t, t0 - 365*DAY, side="left") - 1
#     yoy = p_now - v[i_yoy] if i_yoy >= 0 else np.full(3, np.nan)
#     return anom, yoy
# 
# --- 2. The GPS-Aware Builder ---
# def build_examples_gps(catalog, issue_dates, cells_lat, cells_lon, label_source_catalog, gps_wide, station_map):
#     ev_t = catalog.time.values.astype("datetime64[s]").astype("int64")
#     ev_la, ev_lo, ev_m = catalog.latitude.values, catalog.longitude.values, catalog.magnitude.values
#     lb_t = label_source_catalog.time.values.astype("datetime64[s]").astype("int64")
#     lb_la, lb_lo, lb_m = label_source_catalog.latitude.values, label_source_catalog.longitude.values, label_source_catalog.magnitude.values
# 
#     # GPS prep: per-station sorted lookup for fast causal queries
#     gps_lookup = build_gps_lookup(gps_wide)
# 
#     rows, labels = [], []
#     for d in issue_dates:
#         t0 = np.datetime64(d).astype("datetime64[s]").astype("int64")
#         for i, (clat, clon) in enumerate(zip(cells_lat, cells_lon)):
#             # Seismic Features
#             past = ev_t < t0
#             dpast = haversine_km(clat, clon, ev_la[past], ev_lo[past]) if past.any() else np.array([])
#             tpast = (t0 - ev_t[past]) / DAY if past.any() else np.array([])
#             near = dpast <= RADIUS_KM
#             c7 = int(np.sum(near & (tpast <= 7)))
#             c30 = int(np.sum(near & (tpast <= 30)))
#             c90 = int(np.sum(near & (tpast <= 90)))
#             c365 = int(np.sum(near & (tpast <= 365)))
#             recency = float(tpast[near].min()) if near.any() else 999.0
#             
#             # GPS Features: causal anomaly + year-over-year displacement
#             nearby_stations = station_map[i]
#             anoms, yoys = [], []
#             for sid in nearby_stations:
#                 feats = station_gps_features(gps_lookup, sid, t0)
#                 if feats is not None:
#                     anoms.append(feats[0])
#                     yoys.append(feats[1])
#             if anoms:
#                 gps_feat = np.concatenate([np.nanmean(anoms, axis=0), np.nanmean(yoys, axis=0)])
#                 gps_feat = np.nan_to_num(gps_feat, nan=0.0)
#             else:
#                 gps_feat = np.zeros(6)
# 
#             rows.append([clat, clon, c7, c30, c90, c365, np.log1p(recency), *gps_feat])
# 
#             # Label logic
#             fut = (lb_t > t0) & (lb_t <= t0 + WINDOW_DAYS*DAY)
#             if fut.any():
#                 dfut = haversine_km(clat, clon, lb_la[fut], lb_lo[fut])
#                 hit = np.any((dfut <= RADIUS_KM) & (lb_m[fut] >= TARGET_MAG))
#             else:
#                 hit = False
#             labels.append(int(hit))
#             
#     return np.array(rows, dtype=float), np.array(labels, dtype=int)
# 
# --- 3. Execution Block for GPS Model ---
# 
# Pre-calculate the spatial map
# S_MAP = map_stations_to_cells(CC_LAT, CC_LON, gps_wide)
# 
# print("Building GPS-enhanced TRAIN examples...")
# Xtr_gps, ytr_gps = build_examples_gps(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, gps_wide, S_MAP)
# 
# print("Building GPS-enhanced TEST examples...")
# Xte_gps, yte_gps = build_examples_gps(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, gps_wide, S_MAP)
# 
# print("Building GPS-enhanced JUDGE examples...")
# Xju_gps, yju_gps = build_examples_gps(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, gps_wide, S_MAP)
# 
# Train a completely separate model instance
# model_gps = make_model() 
# model_gps.fit(Xtr_gps, ytr_gps)
# 
# Raw probabilities
# p_test_raw_gps = model_gps.predict_proba(Xte_gps)[:,1]
# p_judge_raw_gps = model_gps.predict_proba(Xju_gps)[:,1]
# 
# Separate calibration for the GPS model
# iso_gps = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_gps, yte_gps)
# p_test_gps = np.clip(iso_gps.predict(p_test_raw_gps), 1e-4, 1-1e-4)
# p_judge_gps = np.clip(iso_gps.predict(p_judge_raw_gps), 1e-4, 1-1e-4)
# 
# print("GPS Model trained and calibrated.")


In [17]:
# --- Coast model: seismic + distance to coast ---
# print("Building COAST examples...")
# Xtr_coast, ytr_coast = build_examples_coast(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, coast_dist)
# Xte_coast, yte_coast = build_examples_coast(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, coast_dist)
# Xju_coast, yju_coast = build_examples_coast(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, coast_dist)
# 
# model_coast = make_model()
# model_coast.fit(Xtr_coast, ytr_coast)
# 
# p_test_raw_coast = model_coast.predict_proba(Xte_coast)[:,1]
# p_judge_raw_coast = model_coast.predict_proba(Xju_coast)[:,1]
# 
# iso_coast = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_coast, yte_coast)
# p_test_coast = np.clip(iso_coast.predict(p_test_raw_coast), 1e-4, 1-1e-4)
# p_judge_coast = np.clip(iso_coast.predict(p_judge_raw_coast), 1e-4, 1-1e-4)
# 
# print("Coast model trained and calibrated.")


In [18]:
# --- Coast+Tide model: seismic + distance to coast + tidal potential ---
# Build tidal lookup: date -> tidal_potential
# df_all = pd.concat([train_raw, test_raw, judge_raw])
# _tidal = df_all[df_all.source == "tidal_ephemeris"]
# _tidal = _tidal[_tidal.parameter == "tidal_potential"][["time", "value"]].copy()
# _tidal = _tidal.sort_values("time").drop_duplicates("time")
# Build tidal lookup: date -> tidal_potential
# tidal_lookup = dict(zip(_tidal.time.dt.date, _tidal.value))
# 
# print("Building COAST+TIDE examples...")
# Xtr_ct, ytr_ct = build_examples_coast_tide(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, coast_dist, tidal_lookup)
# Xte_ct, yte_ct = build_examples_coast_tide(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, coast_dist, tidal_lookup)
# Xju_ct, yju_ct = build_examples_coast_tide(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, coast_dist, tidal_lookup)
# 
# model_ct = make_model()
# model_ct.fit(Xtr_ct, ytr_ct)
# 
# p_test_raw_ct = model_ct.predict_proba(Xte_ct)[:,1]
# p_judge_raw_ct = model_ct.predict_proba(Xju_ct)[:,1]
# 
# iso_ct = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_ct, yte_ct)
# p_test_ct = np.clip(iso_ct.predict(p_test_raw_ct), 1e-4, 1-1e-4)
# p_judge_ct = np.clip(iso_ct.predict(p_judge_raw_ct), 1e-4, 1-1e-4)
# 
# print("Coast+Tide model trained and calibrated.")


In [19]:
# --- Elevation model: seismic + elevation ---
# print("Building ELEVATION examples...")
# Xtr_elev, ytr_elev = build_examples_elevation(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, elevation)
# Xte_elev, yte_elev = build_examples_elevation(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, elevation)
# Xju_elev, yju_elev = build_examples_elevation(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, elevation)
# 
# model_elev = make_model()
# model_elev.fit(Xtr_elev, ytr_elev)
# 
# p_test_raw_elev = model_elev.predict_proba(Xte_elev)[:,1]
# p_judge_raw_elev = model_elev.predict_proba(Xju_elev)[:,1]
# 
# iso_elev = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_elev, yte_elev)
# p_test_elev = np.clip(iso_elev.predict(p_test_raw_elev), 1e-4, 1-1e-4)
# p_judge_elev = np.clip(iso_elev.predict(p_judge_raw_elev), 1e-4, 1-1e-4)
# 
# print("Elevation model trained and calibrated.")


## 4 · Model training

Gradient-boosted classifier on seismic history + coast distance + elevation. Calibrated on test split with isotonic regression.

In [20]:
def make_coast_elev_model():
    """Coast+elevation model with tuned hyperparameters."""
    if HAVE_LGBM:
        return LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                              subsample=0.8, colsample_bytree=0.8, 
                              reg_lambda=5.0, random_state=SEED, verbose=-1)
    else:
        return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
                                               l2_regularization=5.0, random_state=SEED)
        
# --- Coast+Elevation model: seismic + distance to coast + elevation ---
print("Building COAST+ELEV examples...")
Xtr_ce, ytr_ce = build_examples_coast_elev(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, coast_dist, elevation)
Xte_ce, yte_ce = build_examples_coast_elev(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, coast_dist, elevation)
Xju_ce, yju_ce = build_examples_coast_elev(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, coast_dist, elevation)

model_ce = make_model()
model_ce.fit(Xtr_ce, ytr_ce)

p_test_raw_ce = model_ce.predict_proba(Xte_ce)[:,1]
p_judge_raw_ce = model_ce.predict_proba(Xju_ce)[:,1]

iso_ce = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_ce, yte_ce)
p_test_ce = np.clip(iso_ce.predict(p_test_raw_ce), 1e-4, 1-1e-4)
p_judge_ce = np.clip(iso_ce.predict(p_judge_raw_ce), 1e-4, 1-1e-4)

print("Coast+Elevation model trained and calibrated.")

Building COAST+ELEV examples...
Coast+Elevation model trained and calibrated.


In [21]:
# --- Coast+Elevation+Fault model ---
print("Building COAST+ELEV+FAULT examples...")
Xtr_cf, ytr_cf = build_examples_coast_elev_fault(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, coast_dist, elevation, fault_dist)
Xte_cf, yte_cf = build_examples_coast_elev_fault(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, coast_dist, elevation, fault_dist)
Xju_cf, yju_cf = build_examples_coast_elev_fault(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, coast_dist, elevation, fault_dist)

model_cf = make_model()
model_cf.fit(Xtr_cf, ytr_cf)

p_test_raw_cf = model_cf.predict_proba(Xte_cf)[:,1]
p_judge_raw_cf = model_cf.predict_proba(Xju_cf)[:,1]

iso_cf = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_cf, yte_cf)
p_test_cf = np.clip(iso_cf.predict(p_test_raw_cf), 1e-4, 1-1e-4)
p_judge_cf = np.clip(iso_cf.predict(p_judge_raw_cf), 1e-4, 1-1e-4)

print("Coast+Elevation+Fault model trained and calibrated.")


Building COAST+ELEV+FAULT examples...
Coast+Elevation+Fault model trained and calibrated.


In [ ]:
# --- Coast+Elevation+Fault+Dynamic model ---
print("Building COAST+ELEV+FAULT+DYN examples...")
Xtr_cfd, ytr_cfd = build_examples_coast_elev_fault_dyn(ct, monthly("2001-01-01","2021-11-01"), CC_LAT, CC_LON, ct, coast_dist, elevation, fault_dist, ev_fault_dist)
# Each call needs per-event fault distances aligned with its own catalog:
# the builder indexes ev_fault_dist positionally, and cell 17's array covers ct only.
Xte_cfd, yte_cfd = build_examples_coast_elev_fault_dyn(hist_for_test, monthly("2022-01-01","2024-11-01"), CC_LAT, CC_LON, ce, coast_dist, elevation, fault_dist, event_fault_distances(hist_for_test, fault_areas))
Xju_cfd, yju_cfd = build_examples_coast_elev_fault_dyn(hist_for_judge, monthly("2025-01-01","2026-02-01"), CC_LAT, CC_LON, cj, coast_dist, elevation, fault_dist, event_fault_distances(hist_for_judge, fault_areas))

model_cfd = make_model()
model_cfd.fit(Xtr_cfd, ytr_cfd)

p_test_raw_cfd = model_cfd.predict_proba(Xte_cfd)[:,1]
p_judge_raw_cfd = model_cfd.predict_proba(Xju_cfd)[:,1]

iso_cfd = IsotonicRegression(out_of_bounds="clip").fit(p_test_raw_cfd, yte_cfd)
p_test_cfd = np.clip(iso_cfd.predict(p_test_raw_cfd), 1e-4, 1-1e-4)
p_judge_cfd = np.clip(iso_cfd.predict(p_judge_raw_cfd), 1e-4, 1-1e-4)

print("Coast+Elevation+Fault+Dynamic model trained and calibrated.")


Building COAST+ELEV+FAULT+DYN examples...


In [ ]:
# from itertools import product

# # Define the grid
# param_grid = {
#     "max_iter": [200, 400, 800],
#     "learning_rate": [0.01, 0.05, 0.1],
#     "max_depth": [3, 4, 6],
#     "l2_regularization": [0.1, 1.0, 5.0],
# }

# # Generate all combinations
# keys = list(param_grid.keys())
# combinations = list(product(*[param_grid[k] for k in keys]))

# results = []
# for combo in combinations:
#     params = dict(zip(keys, combo))
#     print(f"Trying: {params}")
    
#     # Build and train model
#     from sklearn.ensemble import HistGradientBoostingClassifier
#     model = HistGradientBoostingClassifier(
#         random_state=SEED,
#         **params,
#     )
#     model.fit(Xtr_ce, ytr_ce)
    
#     # Evaluate on test split
#     p_test = model.predict_proba(Xte_ce)[:, 1]
#     p_test = np.clip(p_test, 1e-4, 1 - 1e-4)
    
#     ig = information_gain_bits(yte_ce, p_test, base)
#     auc = roc_auc_score(yte_ce, p_test)
#     results.append({**params, "test_ig": ig, "test_auc": auc})
#     print(f"  test IG: {ig:.4f}, AUC: {auc:.4f}")

# # Show results sorted by test IG
# results_df = pd.DataFrame(results).sort_values("test_ig", ascending=False)
# print("\n=== Top 5 configurations (by test IG) ===")
# print(results_df.head())

## 6 · The curves

ROC, precision–recall, and a reliability (calibration) diagram on the held-out **judge** split.
These are your visual baseline — beat them.

## 6 · Visualization

ROC, precision-recall, and reliability diagrams on the judge split.

In [ ]:
# fig, ax = plt.subplots(1, 3, figsize=(16,4.6))

# # ROC
# fpr, tpr, _ = roc_curve(yju, p_judge)
# ax[0].plot(fpr, tpr, color="#1F6FB2", lw=2, label=f"AUC = {roc_auc_score(yju,p_judge):.3f}")
# ax[0].plot([0,1],[0,1],"--",color="#5B6B7A",lw=1)
# ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
# ax[0].set_title("ROC — judge split"); ax[0].legend(loc="lower right")

# # PR
# prec, rec, _ = precision_recall_curve(yju, p_judge)
# ax[1].plot(rec, prec, color="#2E7D53", lw=2, label=f"PR-AUC = {average_precision_score(yju,p_judge):.3f}")
# ax[1].axhline(yju.mean(), ls="--", color="#5B6B7A", lw=1, label=f"base rate = {yju.mean():.3f}")
# ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
# ax[1].set_title("Precision–Recall — judge split"); ax[1].legend(loc="upper right")

# # Reliability
# frac_pos, mean_pred = calibration_curve(yju, p_judge, n_bins=10, strategy="quantile")
# ax[2].plot(mean_pred, frac_pos, "o-", color="#C0392B", lw=2, label=f"ECE = {ece(yju,p_judge):.3f}")
# ax[2].plot([0,1],[0,1],"--",color="#5B6B7A",lw=1)
# ax[2].set_xlabel("Predicted probability"); ax[2].set_ylabel("Observed frequency")
# ax[2].set_title("Reliability — judge split"); ax[2].legend(loc="upper left")

# plt.tight_layout(); plt.show()# Use the same training base rate for both to ensure a fair IG_bits comparison
# base = ytr.mean() 

# # Generate scorecards for both models
# rows = [
#     scorecard("test_baseline", yte, p_test, base),
#     scorecard("test_gps", yte_gps, p_test_gps, base),
#     scorecard("judge_baseline", yju, p_judge, base),
#     scorecard("judge_gps", yju_gps, p_judge_gps, base),
# ]

# comp_df = pd.DataFrame(rows).set_index("split")
# pd.options.display.float_format = lambda v: f"{v:.4f}"

# print("=== Model Comparison — Accuracy Statistics ===")
# # Reordering columns for easier reading
# cols = ["AUC", "PR_AUC", "IG_bits", "Brier", "ECE", "base_rate", "positives", "n"]
# print(comp_df[cols])


In [ ]:
# Final comparison: baseline vs. coast+elevation model
base = ytr.mean()

rows = [
    scorecard("test_baseline", yte, p_test, base),
    scorecard("test_coast_elev", yte_ce, p_test_ce, base),
    scorecard("test_coast_elev_fault", yte_cf, p_test_cf, base),
    scorecard("test_coast_elev_fault_dyn", yte_cfd, p_test_cfd, base),
    scorecard("judge_baseline", yju, p_judge, base),
    scorecard("judge_coast_elev", yju_ce, p_judge_ce, base),
    scorecard("judge_coast_elev_fault", yju_cf, p_judge_cf, base),
    scorecard("judge_coast_elev_fault_dyn", yju_cfd, p_judge_cfd, base),
]

comp_df = pd.DataFrame(rows).set_index("split")
pd.options.display.float_format = lambda v: f"{v:.4f}"

print("=== Final Model Comparison ===")
cols = ["AUC", "PR_AUC", "IG_bits", "Brier", "ECE", "base_rate", "positives", "n"]
print(comp_df[cols])


## 7 · Your turn

You now have a complete, honest, working forecast. It is **plain on purpose.** Some directions teams
often explore (no promises — you decide what's worth it):

- **Richer seismicity features** — b-values, event-rate acceleration, moment release, multi-radius counts.
- **The other four data streams** — geomagnetic, solar, tidal, GPS. Do they add anything? Prove it, don't assume it.
- **A physics baseline** — the ETAS aftershock model is a classic; feeding its forecast in as a feature is a strong move.
- **Better calibration and model selection** — how you split time and calibrate can matter more than the model.
- **Quantum approaches** — encodings, kernels, variational classifiers, if your track calls for it.

**Two things worth remembering:**
1. Report your gains *with* calibration, and always check your held-out (judge) number against your
   own test number — if judge is far worse, you probably leaked.
2. If your fancy pipeline doesn't clearly beat this simple one, that's a real and interesting result.
   Say so, honestly, with the numbers to back it up.

Good luck. Build something you can defend.

In [ ]:
# import seaborn as sns
# features = [
# "time","latitude","longitude","depth_km","magnitude"
# ]
# corr = df[features].corr()
# sns.heatmap(corr, annot=True, cmap="coolwarm")
# plt.title("Correlation between predictive features")
# plt.show()

In [ ]:
# fig, ax = plt.subplots(1, 3, figsize=(18, 5))
# 
# --- 1. ROC Curve Comparison ---
# Baseline
# fpr_b, tpr_b, _ = roc_curve(yju, p_judge)
# ax[0].plot(fpr_b, tpr_b, color="#1F6FB2", lw=2, label=f"Baseline AUC = {roc_auc_score(yju, p_judge):.3f}")
# 
# GPS Model
# fpr_g, tpr_g, _ = roc_curve(yju_gps, p_judge_gps)
# Coast Model
# fpr_c, tpr_c, _ = roc_curve(yju_coast, p_judge_coast)
# ax[0].plot(fpr_c, tpr_c, color="#2E7D53", lw=2, label=f"Coast AUC = {roc_auc_score(yju_coast, p_judge_coast):.3f}")
# ax[0].plot(fpr_g, tpr_g, color="#C0392B", lw=2, label=f"GPS AUC = {roc_auc_score(yju_gps, p_judge_gps):.3f}")
# 
# ax[0].plot([0,1],[0,1],"--", color="#5B6B7A", lw=1)
# ax[0].set_xlabel("False positive rate")
# ax[0].set_ylabel("True positive rate")
# ax[0].set_title("ROC Comparison — judge split")
# ax[0].legend(loc="lower right")
# 
# --- 2. Precision-Recall Comparison ---
# Baseline
# prec_b, rec_b, _ = precision_recall_curve(yju, p_judge)
# ax[1].plot(rec_b, prec_b, color="#2E7D53", lw=2, label=f"Baseline PR-AUC = {average_precision_score(yju, p_judge):.3f}")
# 
# GPS Model
# prec_g, rec_g, _ = precision_recall_curve(yju_gps, p_judge_gps)
# Coast Model
# prec_c, rec_c, _ = precision_recall_curve(yju_coast, p_judge_coast)
# ax[1].plot(rec_c, prec_c, color="#2E7D53", lw=2, label=f"Coast PR-AUC = {average_precision_score(yju_coast, p_judge_coast):.3f}")
# ax[1].plot(rec_g, prec_g, color="#E67E22", lw=2, label=f"GPS PR-AUC = {average_precision_score(yju_gps, p_judge_gps):.3f}")
# 
# ax[1].axhline(yju.mean(), ls="--", color="#5B6B7A", lw=1, label=f"base rate = {yju.mean():.3f}")
# ax[1].set_xlabel("Recall")
# ax[1].set_ylabel("Precision")
# ax[1].set_title("PR Comparison — judge split")
# ax[1].legend(loc="upper right")
# 
# --- 3. Reliability Comparison ---
# Baseline
# frac_pos_b, mean_pred_b = calibration_curve(yju, p_judge, n_bins=10, strategy="quantile")
# ax[2].plot(mean_pred_b, frac_pos_b, "o-", color="#1F6FB2", lw=2, label=f"Baseline ECE = {ece(yju, p_judge):.3f}")
# 
# GPS Model
# frac_pos_g, mean_pred_g = calibration_curve(yju_gps, p_judge_gps, n_bins=10, strategy="quantile")
# Coast Model
# frac_pos_c, mean_pred_c = calibration_curve(yju_coast, p_judge_coast, n_bins=10, strategy="quantile")
# ax[2].plot(mean_pred_c, frac_pos_c, "o-", color="#2E7D53", lw=2, label=f"Coast ECE = {ece(yju_coast, p_judge_coast):.3f}")
# ax[2].plot(mean_pred_g, frac_pos_g, "o-", color="#C0392B", lw=2, label=f"GPS ECE = {ece(yju_gps, p_judge_gps):.3f}")
# 
# ax[2].plot([0,1],[0,1],"--", color="#5B6B7A", lw=1)
# ax[2].set_xlabel("Predicted probability")
# ax[2].set_ylabel("Observed frequency")
# ax[2].set_title("Reliability Comparison — judge split")
# ax[2].legend(loc="upper left")
# 
# plt.tight_layout()
# plt.show()
